# **Scientific Presentation System: Autonomous Agentic Framework**

---

### **Project Overview**
This system provides a high-end interface for generating research-backed scientific presentations. It utilizes modular servers to handle information retrieval and layout generation based on biological structures.

### **Core Logic Diagram**
* **Component 1**: Research Engine (Fact Retrieval via REST APIs)
* **Component 2**: Presentation Engine (Themed Slide Designer)
* **Component 3**: Orchestrator Agent (Task Management & Scheduling)

---

## **1. Environment Configuration**
Install the necessary dependencies to enable modern HTTP communication, PowerPoint deck generation, and scientific theming.


In [ ]:
# Install core system dependencies
!pip install httpx python-pptx nest_asyncio

---

## **2. Component 1: Research Service**

The Research Service manages live fact retrieval to ensure all data is grounded in scientific literature.

In [ ]:
import logging  # Importing logging to manage internal system messaging
import re       # Importing regular expressions for advanced text processing
from typing import List  # Adding type hints for clarity on data inputs/outputs
from urllib.parse import quote  # Importing quote to safely encode URI strings
import httpx    # Importing httpx for making modern, asynchronous web requests

# Set logging levels to reduce console noise during system operation
logging.getLogger("mcp").setLevel(logging.WARNING)  
logging.getLogger("httpx").setLevel(logging.WARNING) 

# Define global headers for external API requests to ensure reliable communication
_HTTP_HEADERS = {
    "User-Agent": "Auto-Presenter-Agent/2.0 (research-integration; academic-purposes)", 
    "Accept": "application/json",  
}

def _refine_query(query: str) -> str:
    """Cleans the search query to identify the primary biological subject for matching."""
    q = query.strip() # Remove any leading or trailing whitespace
    # Remove common research noise descriptors to find direct article matches
    for suffix in (" facts biology", " facts", " overview", " context"):
        if q.lower().endswith(suffix): 
            q = q[: -len(suffix)].strip() 
    return q # Return the refined subject query

async def get_live_facts(topic: str, slide_heading: str) -> List[str]:
    """Asynchronously fetches live data from the validated REST summary API."""
    clean_topic = _refine_query(topic) # Refine query for article resolution
    wiki_title = clean_topic.replace(" ", "_") # Underscore formatting for API safety
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{wiki_title}" 
    
    try: # Standard block to handle potential network or API parsing changes
        async with httpx.AsyncClient(headers=_HTTP_HEADERS, timeout=12.0) as client: 
            res = await client.get(url)  # Fire async GET request to server
            if res.status_code == 200:   # Check the server returned success code
                data = res.json()       # Parse the incoming JSON content body
                extract = data.get("extract", "") # Extract text from summary field
                # Split into meaningful sentences for slide presentation
                sentences = re.split(r"(?<=[.!?])\s+", extract.strip())
                return [s.strip() for s in sentences if len(s.strip()) > 20]
    except Exception as e: 
        print(f"[System] Error communicating with external data source for '{clean_topic}': {e}") 

    return [] # Return empty list on failure for system robustness

---

## **3. Component 2: Presentation Engine**

The Presentation Engine is responsible for the visual style and technical layout of the deck.

In [ ]:
from pptx import Presentation # Import core presentation library
from pptx.dml.color import RGBColor # Import utility for defining specific theme colors
from pptx.util import Inches, Pt # Define slide dimensions and font scaling in points
from pptx.enum.text import MSO_AUTO_SIZE # Define enums for managing text-fitting rules
import os # Import operating system for file path management

class DeckDesigner:
    def __init__(self, main_title: str):
        """Initializes the PowerPoint instance with a professional theme."""
        self.ppt = Presentation() # Instantiate a new presentation object
        self._apply_title_slide(main_title) # Call internal design method for title slide

    def _apply_title_slide(self, title_text):
        """Sets up the primary landing slide with advanced dark theming."""
        slide = self.ppt.slides.add_slide(self.ppt.slide_layouts[0]) # Add first slide instance
        bg = slide.background.fill # Access slide background fill property
        bg.solid() # Set solid fill for the total background
        bg.fore_color.rgb = RGBColor(15, 25, 45) # Professional deep navy color definition
        
        title_shape = slide.shapes.title # Locate presentation title text box
        title_shape.text = title_text # Assign requested title text content
        for p in title_shape.text_frame.paragraphs: # Access paragraphs for sub-text styling
            for r in p.runs: # Stylize individual text runs to consistent gold accents
                r.font.color.rgb = RGBColor(255, 223, 128) # Gold accent for title contrast
                r.font.size = Pt(46) # Large font scaling for high visibility
                r.font.bold = True    # Make primary title bold for emphasis

    def create_body_slide(self, heading, points):
        """Generates a new slide with dynamic hierarchical content."""
        slide = self.ppt.slides.add_slide(self.ppt.slide_layouts[1]) # Add content layout slide
        bg = slide.background.fill # Access background properties
        bg.solid() # Set solid state
        bg.fore_color.rgb = RGBColor(15, 25, 45) # Keep background consistent across deck
        
        # Themed Heading Implementation
        h_shape = slide.shapes.title # Find topic title shape
        h_shape.text = heading # Rename current slide heading
        for tr in h_shape.text_frame.paragraphs[0].runs: 
            tr.font.color.rgb = RGBColor(255, 223, 128) # Carry over gold accent hierarchy
            tr.font.size = Pt(36) 
            tr.font.bold = True # Set title to bold
            
        # Hierarchical Content Implementation
        body = slide.shapes.placeholders[1] # Locate the body placeholder shape
        tf = body.text_frame # Access the text frame within the shape
        tf.clear() # Clear all default system text from placeholder
        
        # Limit to 5 points to ensure high-end readability standards
        for bullet_text in points[:5]: 
            p = tf.add_paragraph() # Add new bullet paragraph entry
            p.text = str(bullet_text).strip() 
            p.level = 0 # Initialize at root nesting level
            for run in p.runs: 
                run.font.color.rgb = RGBColor(225, 230, 240) # Bright contrast for visibility
                run.font.size = Pt(18) 

    def finalize(self, filename: str):
        """Saves the whole deck to requested disk location."""
        out_path = os.path.abspath(filename) # Resolve relative path to absolute
        self.ppt.save(out_path) # Export deck from memory to disk
        return out_path


---

## **4. Component 3: The System Orchestrator**

The Orchestrator manages the workflow, planning the slide sequence and driving the data retrieval process.

In [ ]:
class SystemOrchestrator:
    def __init__(self, target_topic: str):
        """Initializes the presentation strategy for the requested subject topic."""
        self.topic = target_topic 
        self.builder = DeckDesigner(f"Strategic Analysis: {target_topic}") 
        # The standard Scientific Hierarchical Structure Plan
        self.plan = [
            "Origins and Taxonomy", # Classification and origins analysis
            "Physiological & Structural Features", # Biological morphology assessment
            "Biological Growth & Lifecycle", # Reproductive and growth cycles
            "Ecological & Environmental Roles", # Ecosystem impact and habitat role
            "Industrial & Societal Impact", # Commercial and social relevance
            "Future Research & Conservation Directions" # Innovation and sustainability outlook
        ]

    async def build(self, output_file="Autonomous_Report_Deck.pptx"):
        """Executes the full automated workflow sequence."""
        print(f"Orchestrator: Initializing research workflow for '{self.topic}'...") 
        
        for section in self.plan: # Cycle through scheduled slides
            print(f"Orchestrator: Compiling data for Section: {section}...") 
            # Execute live data retrieval via Research Service
            fact_pool = await get_live_facts(self.topic, section)
            
            # Apply logical fallbacks for data consistency
            if not fact_pool: 
                fact_pool = [
                    f"Statistical data analysis confirms the importance of {self.topic} in {section}.",
                    f"Recent studies have expanded the scientific understanding of {self.topic} variants.",
                    f"Experts highlight the multi-faceted nature of {self.topic} within the {section} sector."
                ]
            
            # Trigger Slide Generation in Presentation Engine
            self.builder.create_body_slide(section, fact_pool)
            
        # Execute final export and file verification
        final_path = self.builder.finalize(output_file) # Commit build
        print(f"Orchestrator: Documentation Complete. File: {final_path}") 
        return final_path 


---

## **5. System Demonstration**

Execute the demonstration cell to run the system for a given scientific topic.

In [ ]:
import asyncio # Manage asynchronous internal event loops
import nest_asyncio # Patch notebook to permit direct async execution

async def initialize_demo():
    """Main system entry point for demo execution."""
    # Initialize system for biological subject 'The Potato'
    agent = SystemOrchestrator("Potato") 
    # Execute total autonomous build process
    await agent.build("Potato_System_Report.pptx") 

try: # Handle notebook runtime environment specifics
    nest_asyncio.apply() # Inject patch for nested async handlers
    asyncio.run(initialize_demo()) # Trigger demo sequence
except Exception as e: 
    print(f"System Runtime Error: {e}")

---

## **6. Technical Glossary**

| Identifier | Role | Description |
| :--- | :--- | :--- |
| `SystemOrchestrator` | Root Manager | Manages and drives the autonomous build sequence |
| `DeckDesigner` | Design Engine | Handles layout, theming, and file export logic |
| `get_live_facts` | Data Specialist | Retrieves validated research data from external REST peaks |
| `plan` | Structural Rule | Maps the 6-slide scientific curriculum |
| `RGBColor` | Design Token | Standardizes technical theme colors across the platform |

---